# Design Decisions

## Initial Rating
- Every new team starts at 1500.

## Unknown Teams
- Automatically initialize new teams.

## Training Period
- 1 January 2000 onwards.

## Training Matches
- Only completed matches.

In [1]:
ratings = {}

def initialize_team(team):
    if team not in ratings:
        ratings[team] = 1500

# Day 6 - Building the Elo Rating System

## Goal

Today we begin implementing our own Elo rating engine.

Objectives:

- Understand expected probability
- Implement the Elo expected score formula
- Test the formula on sample teams

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sys
sys.path.append("..")

In [29]:
from config.k_factors import K_FACTORS

In [4]:
matches = pd.read_csv(
    r"E:\Python\worldcup-intelligence-platform\data\processed\clean_matches.csv",
    parse_dates=["date"]
)

In [5]:
initialize_team("Brazil")
initialize_team("Argentina")
initialize_team("Brazil")

ratings

{'Brazil': 1500, 'Argentina': 1500}

In [6]:
def expected_score(home_team_rating, away_team_rating):
    expected = 1 / (1 + 10 ** ((away_team_rating - home_team_rating) / 400))
    return expected

In [7]:
print(expected_score(1500, 1500))

0.5


In [8]:
print(expected_score(1700, 1500))

0.7597469266479578


In [9]:
print(expected_score(1500, 1700))

0.2402530733520421


In [10]:
print(expected_score(2000, 1500))

0.9467597847979775


In [11]:
def actual_score(home_score, away_score):
    if home_score > away_score:
        return 1, 0

    elif away_score > home_score:
        return 0, 1

    else:
        return 0.5, 0.5

In [12]:
print(actual_score(3, 0))

(1, 0)


In [13]:
print(actual_score(1, 1))

(0.5, 0.5)


In [14]:
print(actual_score(0, 2))

(0, 1)


In [30]:
def update_elo(team_rating, expected, actual, k = get_k_factor(row["tournament"])):
    new_rating = team_rating + k*(actual - expected)
    return new_rating

In [31]:
ratings = {}

for _, row in matches.iterrows():

    # Initialize teams
    initialize_team(row["home_team"])
    initialize_team(row["away_team"])

    # Current ratings
    home_rating = ratings[row["home_team"]]
    away_rating = ratings[row["away_team"]]

    # Expected scores
    home_expected = expected_score(home_rating, away_rating)
    away_expected = expected_score(away_rating, home_rating)

    # Actual scores
    home_actual, away_actual = actual_score(
        row["home_score"],
        row["away_score"]
    )

    # Updated ratings
    home_new = update_elo(
        home_rating,
        home_expected,
        home_actual
    )

    away_new = update_elo(
        away_rating,
        away_expected,
        away_actual
    )

    # Save back
    ratings[row["home_team"]] = home_new
    ratings[row["away_team"]] = away_new

In [32]:
len(ratings)

321

In [33]:
ratings

{'Egypt': 1836.2562443197778,
 'Togo': 1529.02810678495,
 'Tunisia': 1648.8574093076707,
 'Trinidad and Tobago': 1484.4633775067302,
 'Canada': 1842.7202869454954,
 'Mexico': 2003.6295296799535,
 'Iran': 1839.3926552830208,
 'Ivory Coast': 1882.2337341775615,
 'Burkina Faso': 1632.6996305498699,
 'Gabon': 1505.9362729873726,
 'Guatemala': 1598.9812076334365,
 'Armenia': 1426.4794733209517,
 'Bermuda': 1379.4354229412409,
 'Cameroon': 1695.5639961341833,
 'Senegal': 1819.317700706767,
 'China': 1566.8077295324636,
 'New Zealand': 1630.1064266614153,
 'Jamaica': 1606.8329997369153,
 'United States': 1894.6969863778136,
 'Morocco': 2017.3902649943516,
 'Ghana': 1645.2066392143447,
 'Panama': 1694.4951665793594,
 'Algeria': 1905.8593638967372,
 'Malta': 1449.9073255993308,
 'Qatar': 1530.5643137120855,
 'South Korea': 1810.7361038052322,
 'Philippines': 1407.3715245136846,
 'Zambia': 1501.815917270927,
 'Nigeria': 1837.6080932310651,
 'South Africa': 1703.9964380151637,
 'Vietnam': 1547.75

In [34]:
list(ratings.items())[:10]

[('Egypt', 1836.2562443197778),
 ('Togo', 1529.02810678495),
 ('Tunisia', 1648.8574093076707),
 ('Trinidad and Tobago', 1484.4633775067302),
 ('Canada', 1842.7202869454954),
 ('Mexico', 2003.6295296799535),
 ('Iran', 1839.3926552830208),
 ('Ivory Coast', 1882.2337341775615),
 ('Burkina Faso', 1632.6996305498699),
 ('Gabon', 1505.9362729873726)]

In [35]:
sorted_ratings = sorted(ratings.items(), key=lambda x: x[1], reverse=True)

In [36]:
sorted_ratings[:20]

[('Argentina', 2123.091484227609),
 ('Spain', 2091.5221081240684),
 ('France', 2054.9138488857266),
 ('Morocco', 2017.3902649943516),
 ('Brazil', 2009.7549284145784),
 ('Portugal', 2008.1339010174356),
 ('Mexico', 2003.6295296799535),
 ('Colombia', 1996.065059801437),
 ('Germany', 1994.6510855534805),
 ('England', 1982.9527308554684),
 ('Netherlands', 1982.8658513276407),
 ('Japan', 1980.3220391733284),
 ('Norway', 1971.2747055605525),
 ('Ecuador', 1946.815072916353),
 ('Switzerland', 1928.300148028561),
 ('Turkey', 1926.879699225753),
 ('Italy', 1920.979561363593),
 ('Croatia', 1908.9546874130153),
 ('Algeria', 1905.8593638967372),
 ('Belgium', 1901.6145754589224)]

# Baseline Elo Model (Version 1)

## Assumptions

- Every team starts at 1500.
- K-factor = 40 for every match.
- Goal difference is ignored.
- Every tournament has equal importance.
- Matches are processed chronologically.
- Elo ratings are updated after every match.

## Limitations

- Friendlies count the same as World Cup Finals.
- Winning 1–0 and 8–0 have the same effect.
- Teams do not have historical ratings before the year 2000.

In [37]:
matches = pd.read_csv(
    "../data/processed/clean_matches.csv",
    parse_dates=["date"]
)

In [38]:
K_FACTORS = {

    # FIFA
    "Friendly": 20,
    "FIFA World Cup qualification": 40,
    "FIFA World Cup": 60,
    "Confederations Cup": 55,

    # Continental Championships
    "UEFA Euro": 50,
    "Copa América": 50,
    "African Cup of Nations": 50,
    "AFC Asian Cup": 50,
    "Gold Cup": 50,
    "Oceania Nations Cup": 50,

    # Continental Qualifiers
    "UEFA Euro qualification": 40,
    "African Cup of Nations qualification": 40,
    "AFC Asian Cup qualification": 40,
    "Gold Cup qualification": 40,
    "Oceania Nations Cup qualification": 40,

    # Nations League
    "UEFA Nations League": 35,
    "CONCACAF Nations League": 35,

    # Regional Cups
    "AFF Championship": 30,
    "ASEAN Championship": 30,
    "Gulf Cup": 30,
    "SAFF Cup": 30,
    "EAFF Championship": 30,
    "WAFF Championship": 30,
    "UNCAF Cup": 30,
    "CECAFA Cup": 30,
    "COSAFA Cup": 30,
    "CFU Caribbean Cup": 30,
    "Arab Cup": 30,

    # Regional Qualifiers
    "AFF Championship qualification": 25,
    "ASEAN Championship qualification": 25,
    "EAFF Championship qualification": 25,
    "Arab Cup qualification": 25,
    "CFU Caribbean Cup qualification": 25,
    "CONCACAF Nations League qualification": 25,
    "COSAFA Cup qualification": 25,

}

In [39]:
def get_k_factor(tournament):
    return K_FACTORS.get(tournament, 20)

In [40]:
matches["tournament"].value_counts().head(30)

tournament
Friendly                                8448
FIFA World Cup qualification            5840
UEFA Euro qualification                 1532
African Cup of Nations qualification    1416
UEFA Nations League                      658
AFC Asian Cup qualification              530
African Cup of Nations                   525
FIFA World Cup                           444
CONCACAF Nations League                  422
Gold Cup                                 359
CECAFA Cup                               341
COSAFA Cup                               326
CFU Caribbean Cup qualification          293
Island Games                             283
UEFA Euro                                277
AFC Asian Cup                            256
AFF Championship                         251
Copa América                             248
Gulf Cup                                 189
SAFF Cup                                 135
EAFF Championship                        130
WAFF Championship                        114